# 06 - Maintenance Block Optimization
## RailOpt AI: Railway Maintenance Planning

## Section 1: Project Introduction

**Prediction** estimates an outcome for each asset: here, the trained model estimates maintenance priority. **Optimization** chooses a feasible set of maintenance jobs, start times, and crews under operational constraints. A high predicted priority is evidence for planning, not a schedule by itself.

Optimization is needed to balance asset risk against finite crews, shared track access, time windows, and train-service disruption. CP-SAT turns the priorities into an auditable block plan that engineers can review and revise. This notebook uses the saved model for inference only; it does not retrain the model or predict failures.

---
## Section 2: Import Libraries
Import data handling, model persistence, plotting, and Google OR-Tools CP-SAT components.

In [ ]:
from math import ceil
from pathlib import Path
import time

import joblib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from ortools.sat.python import cp_model

sns.set_theme(style="whitegrid", context="notebook")
current_dir = Path.cwd()
PROJECT_ROOT = current_dir if (current_dir / "data").exists() else current_dir.parent
MODEL_PATH = PROJECT_ROOT / "outputs" / "models" / "best_model.pkl"
FEATURE_COLUMNS_PATH = PROJECT_ROOT / "outputs" / "models" / "feature_columns.pkl"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "railway_synthetic_training_dataset_7000.xlsx"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TARGET = "priority_score"
print(f"Project root: {PROJECT_ROOT}")
print(f"OR-Tools version: {getattr(cp_model, '__file__', 'installed')}")

---
## Section 3: Load Model and Predict Priority
Load the Stage 4 model/schema and processed test split. Stage 3 omitted identifiers from model inputs, so recover test-row metadata from the raw workbook using the same fixed split seed. This mapping is accepted only when raw row counts and split counts prove that Stage 3 removed no rows.

In [ ]:
required_paths = [
    MODEL_PATH,
    FEATURE_COLUMNS_PATH,
    PROCESSED_DIR / "train.csv",
    PROCESSED_DIR / "validation.csv",
    PROCESSED_DIR / "test.csv",
    RAW_DATA_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required model/data artifacts are missing: "
        + ", ".join(missing_paths)
        + ". Run preprocessing and model training first."
    )

best_model = joblib.load(MODEL_PATH)
feature_columns = joblib.load(FEATURE_COLUMNS_PATH)
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
raw_df = pd.read_excel(RAW_DATA_PATH)
split_sizes = {
    split_name: len(
        pd.read_csv(PROCESSED_DIR / f"{split_name}.csv", usecols=[TARGET])
    )
    for split_name in ("train", "validation", "test")
}
if TARGET not in test_df.columns:
    raise KeyError(f"'{TARGET}' is missing from test.csv.")
if len(raw_df) != sum(split_sizes.values()):
    raise ValueError(
        "Cannot safely align asset IDs to test.csv: raw rows do not match the "
        "processed split sizes. Re-export the splits with asset_id and station metadata."
    )

# Stage 3 shuffled the 30% holdout again when splitting it into validation and test.
split_random_state = 42
permutation = np.random.RandomState(split_random_state).permutation(len(raw_df))
holdout_count = ceil(0.30 * len(raw_df))
holdout_indices = permutation[:holdout_count]
test_count = ceil(0.50 * holdout_count)
second_split_order = np.random.RandomState(split_random_state).permutation(
    len(holdout_indices)
)
test_source_indices = holdout_indices[second_split_order[:test_count]]
if test_count != len(test_df):
    raise ValueError(
        "Test split size differs from the Stage 3 split expected by this metadata mapping."
    )
raw_test_metadata = raw_df.iloc[test_source_indices].reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
if len(raw_test_metadata) != len(test_df):
    raise ValueError("Raw asset metadata and processed test rows could not be aligned.")
missing_features = sorted(set(feature_columns) - set(test_df.columns))
if missing_features:
    raise KeyError(f"test.csv is missing model features: {missing_features}")

X_test = test_df.loc[:, feature_columns]
predicted_priority = best_model.predict(X_test)
if not np.isfinite(predicted_priority).all():
    raise ValueError("Model produced non-finite priority predictions.")

optimization_df = raw_test_metadata.drop(columns=[TARGET], errors="ignore").copy()
optimization_df["predicted_priority_score"] = predicted_priority
print(f"Model: {type(best_model).__name__}")
print(f"Assets predicted: {len(optimization_df):,}")
print(
    f"Test feature count: {len(feature_columns)}; "
    f"raw metadata rows aligned: {len(raw_test_metadata):,}"
)
display(
    optimization_df[
        ["asset_id", "asset_type", "station", "predicted_priority_score"]
    ].head()
)

---
## Section 4: Generate Maintenance Requests
Create one request for every test asset whose predicted priority is strictly greater than the configurable threshold. Duration and resource context come from raw operational units, not the standardized model matrix. One crew is assigned per request; the shift crew pool is configured in Section 5.

In [ ]:
REQUEST_PRIORITY_THRESHOLD = 75.0
CRITICAL_PRIORITY_THRESHOLD = 95.0
required_request_columns = [
    "asset_id",
    "asset_type",
    "station",
    "predicted_priority_score",
    "maintenance_duration",
    "crew_available",
    "passenger_peak",
    "nearby_alternate_track",
    "train_frequency",
]
missing_request_columns = sorted(
    set(required_request_columns) - set(optimization_df.columns)
)
if missing_request_columns:
    raise KeyError(f"Raw data lacks scheduling fields: {missing_request_columns}")

maintenance_requests = optimization_df.loc[
    optimization_df["predicted_priority_score"] > REQUEST_PRIORITY_THRESHOLD
].copy()
maintenance_requests["Maintenance ID"] = [
    f"MNT-{asset_id}" for asset_id in maintenance_requests["asset_id"]
]
maintenance_requests["Estimated Maintenance Duration"] = (
    maintenance_requests["maintenance_duration"]
    .map(lambda value: max(1, ceil(float(value))))
    .astype(int)
)
maintenance_requests["Required Crew"] = 1
maintenance_requests["Preferred Maintenance Window"] = np.where(
    maintenance_requests["predicted_priority_score"] >= CRITICAL_PRIORITY_THRESHOLD,
    "Overnight 20:00-08:00; critical daytime 00:00-12:00 allowed",
    "Overnight 20:00-08:00",
)
maintenance_requests["Track Availability"] = maintenance_requests[
    "nearby_alternate_track"
].map(
    lambda value: "Available"
    if str(value).strip().lower() in {"yes", "true", "1"}
    else "Unavailable"
)
maintenance_requests["Critical"] = (
    maintenance_requests["predicted_priority_score"]
    >= CRITICAL_PRIORITY_THRESHOLD
)
maintenance_requests = maintenance_requests.sort_values(
    ["predicted_priority_score", "asset_id"], ascending=[False, True]
).reset_index(drop=True)

request_columns = [
    "Maintenance ID",
    "asset_id",
    "asset_type",
    "station",
    "predicted_priority_score",
    "Estimated Maintenance Duration",
    "Required Crew",
    "Preferred Maintenance Window",
    "Track Availability",
]
print(f"Priority threshold: > {REQUEST_PRIORITY_THRESHOLD:.1f}")
print(f"Critical priority threshold: >= {CRITICAL_PRIORITY_THRESHOLD:.1f}")
print(f"Eligible maintenance requests: {len(maintenance_requests):,}")
display(maintenance_requests[request_columns].head(10))

In [ ]:
if maintenance_requests.empty:
    raise ValueError(
        "No test assets exceed the configured priority threshold. "
        "Lower REQUEST_PRIORITY_THRESHOLD to create an optimization problem."
    )

---
## Section 5: Optimization Constraints

The CP-SAT model enforces: (1) each selected job consumes its configured required crew count (one for this dataset); (2) a cumulative resource constraint caps concurrent crew use at `MAX_CREWS_PER_SHIFT`; (3) jobs at the same station cannot overlap, using station as a conservative track-group proxy because no track ID is present; (4) each job must fit fully within an allowed maintenance window; (5) higher predicted priorities receive larger service and early-completion rewards; (6) critical jobs (priority at least the configured threshold) are mandatory and must start within 24 hours; (7) peak-hour overlap receives a disruption penalty; and (8) jobs without an alternate track receive an additional penalty.

Normal requests may use overnight windows (20:00-08:00). Critical jobs may also use the emergency daytime window (00:00-12:00) to make the deadline feasible. Peak passenger hours are configurable. Peak overlap is penalized rather than forbidden so a critical job can meet its deadline when no fully off-peak window is feasible. If mandatory critical requests cannot all fit the crew, station, window, and deadline constraints, CP-SAT returns infeasible instead of silently violating a safety requirement.

In [ ]:
# Confirm all operational settings with the local railway control office before deployment.
MAX_CREWS_PER_SHIFT = 20  # Initial planning assumption; source crew_available peaks at 20.
HORIZON_DAYS = 7
CRITICAL_DEADLINE_HOURS = 24
NIGHT_WINDOW_START_HOUR = 20
NIGHT_WINDOW_END_HOUR = 8
CRITICAL_DAY_WINDOW = (0, 12)
PEAK_PASSENGER_WINDOWS = ((7, 10), (16, 20))
SOLVER_TIME_LIMIT_SECONDS = 60
PLANNING_START = pd.Timestamp.now().ceil("h")
HORIZON_HOURS = HORIZON_DAYS * 24


def is_within_night_window(timestamp):
    """Return whether an hour falls within the configured overnight block."""
    return (
        timestamp.hour >= NIGHT_WINDOW_START_HOUR
        or timestamp.hour < NIGHT_WINDOW_END_HOUR
    )


def is_within_critical_day_window(timestamp):
    """Return whether an hour falls within the critical-only daytime block."""
    start_hour, end_hour = CRITICAL_DAY_WINDOW
    return start_hour <= timestamp.hour < end_hour


def peak_overlap_count(start_offset, duration_hours):
    """Count occupied hourly slots intersecting configured passenger peaks."""
    occupied_times = [
        PLANNING_START + pd.Timedelta(hours=offset)
        for offset in range(start_offset, start_offset + duration_hours)
    ]
    return sum(
        any(start <= timestamp.hour < end for start, end in PEAK_PASSENGER_WINDOWS)
        for timestamp in occupied_times
    )


def candidate_window_starts(duration_hours, is_critical):
    """Enumerate hourly starts that keep the full job inside an allowed window."""
    candidates = {}
    last_start = HORIZON_HOURS - duration_hours
    for start_offset in range(last_start + 1):
        occupied_times = [
            PLANNING_START + pd.Timedelta(hours=offset)
            for offset in range(start_offset, start_offset + duration_hours)
        ]
        if all(is_within_night_window(timestamp) for timestamp in occupied_times):
            candidates[start_offset] = (
                peak_overlap_count(start_offset, duration_hours),
                0,
            )
        elif is_critical and all(
            is_within_critical_day_window(timestamp) for timestamp in occupied_times
        ):
            candidates[start_offset] = (
                peak_overlap_count(start_offset, duration_hours),
                1,
            )
    return candidates


print(f"Planning starts: {PLANNING_START}; horizon: {HORIZON_DAYS} days")
print(f"Crew pool: {MAX_CREWS_PER_SHIFT}; peak windows: {PEAK_PASSENGER_WINDOWS}")

---
## Section 6: Weighted Optimization Objective

The maximized integer objective combines these configurable terms: **1,000,000 points per scheduled asset** (availability); **10,000 per priority point** (risk coverage); **250 per priority point per hour earlier than the horizon** (early service); **100 points per estimated train exposed** (disruption); **100 reward points per assigned crew-hour** (reduces fleet idle time); **25 cost points per crew-hour** (maintenance cost); **50,000 points for no alternate track**; **10,000 per peak-overlap hour**, doubled for passenger-peak records; and **10,000 for using the critical daytime window** (overnight preference).

The train-impact estimate is `train_frequency * duration / 24`; it is a proxy, not observed train delay. CP-SAT maximizes total weighted benefit minus penalties. The constants are planning assumptions and must be calibrated against railway operating costs and safety policy.

---
## Section 7: Solve Optimization Problem
Build optional maintenance intervals, crew assignments, station-level track conflict groups, deadline rules, and the weighted objective in OR-Tools CP-SAT. A fixed single-worker seed makes the solve reproducible; the schedule is chosen by the optimizer, never randomized.

In [ ]:
AVAILABILITY_REWARD_WEIGHT = 1_000_000
PRIORITY_REWARD_WEIGHT = 10_000
EARLY_SERVICE_WEIGHT = 250
TRAIN_DELAY_PENALTY_WEIGHT = 100
CREW_IDLE_REWARD_WEIGHT = 100
MAINTENANCE_COST_PENALTY_WEIGHT = 25
NO_ALTERNATE_TRACK_PENALTY = 50_000
PEAK_OVERLAP_PENALTY_WEIGHT = 10_000
CRITICAL_DAY_WINDOW_PENALTY = 10_000

model = cp_model.CpModel()
task_variables = []
station_intervals = {}
all_intervals = []
all_demands = []
objective_terms = []
unassigned_start = HORIZON_HOURS + 1

for task_index, request in maintenance_requests.iterrows():
    duration = int(request["Estimated Maintenance Duration"])
    required_crew = int(request["Required Crew"])
    priority = int(round(float(request["predicted_priority_score"])))
    critical = bool(request["Critical"])
    if required_crew < 1 or required_crew > MAX_CREWS_PER_SHIFT:
        raise ValueError(
            f"Request {request['Maintenance ID']} has invalid required crew count "
            f"{required_crew}."
        )

    candidate_metadata = candidate_window_starts(duration, critical)
    candidate_starts = sorted(candidate_metadata)
    start_domain_values = sorted(set([unassigned_start] + candidate_starts))
    scheduled = model.NewBoolVar(f"scheduled_{task_index}")
    start = model.NewIntVarFromDomain(
        cp_model.Domain.FromValues(start_domain_values), f"start_{task_index}"
    )
    end = model.NewIntVar(0, HORIZON_HOURS, f"end_{task_index}")
    peak_hours = model.NewIntVar(0, duration, f"peak_hours_{task_index}")
    critical_day_window = model.NewBoolVar(f"day_window_{task_index}")

    allowed_tuples = {(unassigned_start, 0, 0)}
    allowed_tuples.update(
        (candidate_start, peak_count, daytime)
        for candidate_start, (peak_count, daytime) in candidate_metadata.items()
    )
    model.AddAllowedAssignments(
        [start, peak_hours, critical_day_window], sorted(allowed_tuples)
    )

    interval = model.NewOptionalIntervalVar(
        start, duration, end, scheduled, f"maintenance_{task_index}"
    )
    all_intervals.append(interval)
    all_demands.append(required_crew)
    station_intervals.setdefault(str(request["station"]), []).append(interval)
    model.Add(start == unassigned_start).OnlyEnforceIf(scheduled.Not())
    model.Add(end == 0).OnlyEnforceIf(scheduled.Not())
    model.Add(peak_hours == 0).OnlyEnforceIf(scheduled.Not())
    model.Add(critical_day_window == 0).OnlyEnforceIf(scheduled.Not())

    if critical:
        model.Add(scheduled == 1)
        model.Add(start < CRITICAL_DEADLINE_HOURS)

    has_alternate = (
        str(request["nearby_alternate_track"]).strip().lower()
        in {"yes", "true", "1"}
    )
    passenger_peak = (
        str(request["passenger_peak"]).strip().lower() in {"yes", "true", "1"}
    )
    train_frequency = max(0.0, float(request["train_frequency"]))
    estimated_train_impacts = int(ceil(train_frequency * duration / 24.0))
    peak_penalty = PEAK_OVERLAP_PENALTY_WEIGHT * (2 if passenger_peak else 1)

    objective_terms.extend(
        [
            AVAILABILITY_REWARD_WEIGHT * scheduled,
            PRIORITY_REWARD_WEIGHT * priority * scheduled,
            EARLY_SERVICE_WEIGHT * priority * (HORIZON_HOURS * scheduled - end),
            CREW_IDLE_REWARD_WEIGHT * duration * required_crew * scheduled,
            -TRAIN_DELAY_PENALTY_WEIGHT * estimated_train_impacts * scheduled,
            -MAINTENANCE_COST_PENALTY_WEIGHT * duration * required_crew * scheduled,
            -(0 if has_alternate else NO_ALTERNATE_TRACK_PENALTY) * scheduled,
            -peak_penalty * peak_hours,
            -CRITICAL_DAY_WINDOW_PENALTY * critical_day_window,
        ]
    )
    task_variables.append(
        {
            "request_index": task_index,
            "request": request,
            "duration": duration,
            "required_crew": required_crew,
            "priority": priority,
            "estimated_train_impacts": estimated_train_impacts,
            "scheduled": scheduled,
            "start": start,
            "end": end,
            "peak_hours": peak_hours,
            "critical_day_window": critical_day_window,
        }
    )

for station, intervals in station_intervals.items():
    if len(intervals) > 1:
        model.AddNoOverlap(intervals)
if all_intervals:
    model.AddCumulative(all_intervals, all_demands, MAX_CREWS_PER_SHIFT)

model.Maximize(sum(objective_terms))
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = SOLVER_TIME_LIMIT_SECONDS
solver.parameters.num_search_workers = 1
solver.parameters.random_seed = 42
solve_started = time.perf_counter()
solver_status = solver.Solve(model)
solver_seconds = time.perf_counter() - solve_started
solver_status_name = solver.StatusName(solver_status)
if solver_status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    raise RuntimeError(
        f"CP-SAT did not find a feasible schedule. Status: {solver_status_name}. "
        "Mandatory critical jobs may conflict with the configured crew pool, "
        "station blocks, windows, or deadline. Review the operational settings."
    )
print(f"CP-SAT status: {solver_status_name}; solve time: {solver_seconds:.2f}s")
print(f"Objective value: {solver.ObjectiveValue():,.0f}")

---
## Section 8: Generate Schedule Table
Extract CP-SAT assignments into a readable schedule. Requests that cannot fit the horizon or constraints remain visible with a non-scheduled status so they can be escalated rather than silently discarded.

In [ ]:
schedule_records = []
for task in task_variables:
    request = task["request"]
    is_scheduled = solver.Value(task["scheduled"]) == 1
    if is_scheduled:
        start_offset = solver.Value(task["start"])
        end_offset = solver.Value(task["end"])
        peak_hours_value = solver.Value(task["peak_hours"])
        daytime_value = solver.Value(task["critical_day_window"])
        start_time = PLANNING_START + pd.Timedelta(hours=start_offset)
        end_time = PLANNING_START + pd.Timedelta(hours=end_offset)
        priority = task["priority"]
        duration = task["duration"]
        required_crew = task["required_crew"]
        score = (
            AVAILABILITY_REWARD_WEIGHT
            + PRIORITY_REWARD_WEIGHT * priority
            + EARLY_SERVICE_WEIGHT * priority * (HORIZON_HOURS - end_offset)
            + CREW_IDLE_REWARD_WEIGHT * duration * required_crew
            - TRAIN_DELAY_PENALTY_WEIGHT * task["estimated_train_impacts"]
            - MAINTENANCE_COST_PENALTY_WEIGHT * duration * required_crew
            - (
                0
                if str(request["nearby_alternate_track"]).strip().lower()
                in {"yes", "true", "1"}
                else NO_ALTERNATE_TRACK_PENALTY
            )
            - PEAK_OVERLAP_PENALTY_WEIGHT
            * (
                2
                if str(request["passenger_peak"]).strip().lower()
                in {"yes", "true", "1"}
                else 1
            )
            * peak_hours_value
            - CRITICAL_DAY_WINDOW_PENALTY * daytime_value
        )
        status = "Scheduled"
    else:
        start_offset = None
        end_offset = None
        peak_hours_value = 0
        start_time = pd.NaT
        end_time = pd.NaT
        score = 0
        status = "Not scheduled within horizon/constraints"

    schedule_records.append(
        {
            "Maintenance ID": request["Maintenance ID"],
            "Asset": request["asset_id"],
            "Station": request["station"],
            "Start Time": start_time,
            "End Time": end_time,
            "Assigned Crew": None,
            "Predicted Priority": request["predicted_priority_score"],
            "Optimization Score": int(score),
            "Status": status,
            "Start Offset Hours": start_offset,
            "Duration Hours": task["duration"],
            "Required Crew": task["required_crew"],
            "Estimated Train Impacts": (
                task["estimated_train_impacts"] if is_scheduled else 0
            ),
            "Peak Overlap Hours": peak_hours_value,
            "Track Availability": request["Track Availability"],
        }
    )

optimized_schedule = pd.DataFrame(schedule_records)
if not optimized_schedule.empty:
    # Identical crews are interchangeable; label the optimized intervals deterministically.
    crew_free_at = [0] * MAX_CREWS_PER_SHIFT
    scheduled_indices = optimized_schedule.loc[
        optimized_schedule["Status"] == "Scheduled"
    ].sort_values(["Start Offset Hours", "Predicted Priority"], ascending=[True, False]).index
    for row_index in scheduled_indices:
        row = optimized_schedule.loc[row_index]
        start_offset = int(row["Start Offset Hours"])
        end_offset = start_offset + int(row["Duration Hours"])
        required_crew = int(row["Required Crew"])
        available_crews = [
            crew_index
            for crew_index, free_at in enumerate(crew_free_at)
            if free_at <= start_offset
        ]
        if len(available_crews) < required_crew:
            raise RuntimeError("Could not assign non-overlapping crew labels to the CP-SAT intervals.")
        assigned_crews = available_crews[:required_crew]
        for crew_index in assigned_crews:
            crew_free_at[crew_index] = end_offset
        optimized_schedule.at[row_index, "Assigned Crew"] = ", ".join(
            f"Crew-{crew_index + 1:02d}" for crew_index in assigned_crews
        )

    optimized_schedule = optimized_schedule.sort_values(
        ["Status", "Start Time", "Predicted Priority"],
        ascending=[True, True, False],
        na_position="last",
    ).reset_index(drop=True)

schedule_display_columns = [
    "Maintenance ID",
    "Asset",
    "Station",
    "Start Time",
    "End Time",
    "Assigned Crew",
    "Predicted Priority",
    "Optimization Score",
    "Status",
]
display(optimized_schedule[schedule_display_columns].head(30))
print(
    f"Scheduled: {(optimized_schedule['Status'] == 'Scheduled').sum():,} / "
    f"{len(maintenance_requests):,} requests"
)

---
## Section 9: Schedule Visualizations
Save a Gantt chart, crew utilization chart, priority distribution, and maintenance timeline. The Gantt display is capped to the first 60 scheduled jobs by start time for readability; the full schedule remains in the CSV.

In [ ]:
scheduled_rows = optimized_schedule.loc[optimized_schedule["Status"] == "Scheduled"].copy()

if not scheduled_rows.empty:
    gantt_rows = scheduled_rows.sort_values("Start Time").head(60).copy()
    gantt_rows = gantt_rows.sort_values("Start Time", ascending=True)
    fig, ax = plt.subplots(figsize=(13, max(5, min(16, len(gantt_rows) * 0.28))))
    priority_norm = plt.Normalize(gantt_rows["Predicted Priority"].min(), gantt_rows["Predicted Priority"].max() or 1)
    colors = plt.cm.viridis(priority_norm(gantt_rows["Predicted Priority"]))
    for y_position, (_, row) in enumerate(gantt_rows.iterrows()):
        left = mdates.date2num(row["Start Time"])
        width = (row["End Time"] - row["Start Time"]).total_seconds() / 86400
        ax.barh(y_position, width, left=left, height=0.68, color=colors[y_position])
    ax.set_yticks(range(len(gantt_rows)), gantt_rows["Asset"].astype(str))
    ax.invert_yaxis()
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d %H:%M"))
    ax.set(title="Optimized Maintenance Block Schedule", xlabel="Start / end time", ylabel="Asset")
    fig.autofmt_xdate()
    fig.tight_layout()
    gantt_path = FIGURES_DIR / "optimization_gantt_chart.png"
    fig.savefig(gantt_path, dpi=180, bbox_inches="tight")
    plt.show()

    crew_hours = scheduled_rows.groupby("Assigned Crew")["Duration Hours"].sum().sort_index()
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(x=crew_hours.index, y=crew_hours.values, color="#287C8E", ax=ax)
    ax.axhline(HORIZON_HOURS, color="#C44E52", linestyle="--", label="Horizon capacity")
    ax.set(title="Crew Utilization", xlabel="Crew", ylabel="Assigned hours")
    ax.legend()
    fig.tight_layout()
    crew_path = FIGURES_DIR / "optimization_crew_utilization.png"
    fig.savefig(crew_path, dpi=180, bbox_inches="tight")
    plt.show()

    daily_timeline = scheduled_rows.set_index("Start Time").resample("D")["Maintenance ID"].count()
    fig, ax = plt.subplots(figsize=(10, 5))
    daily_timeline.plot(kind="bar", color="#55A868", ax=ax)
    ax.set(title="Scheduled Maintenance Timeline", xlabel="Date", ylabel="Jobs starting")
    fig.tight_layout()
    timeline_path = FIGURES_DIR / "optimization_maintenance_timeline.png"
    fig.savefig(timeline_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {gantt_path}, {crew_path}, {timeline_path}")
else:
    gantt_path = crew_path = timeline_path = None
    print("No jobs were scheduled; schedule charts are skipped.")

fig, ax = plt.subplots(figsize=(9, 5))
if not maintenance_requests.empty:
    sns.histplot(maintenance_requests["predicted_priority_score"], bins=20, color="#4C72B0", label="All requests", ax=ax)
if not scheduled_rows.empty:
    sns.histplot(scheduled_rows["Predicted Priority"], bins=20, color="#DD8452", alpha=0.55, label="Scheduled", ax=ax)
ax.set(title="Predicted Priority Distribution", xlabel="Predicted priority score", ylabel="Asset count")
ax.legend()
fig.tight_layout()
priority_path = FIGURES_DIR / "optimization_priority_distribution.png"
fig.savefig(priority_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {priority_path}")

---
## Section 10: Optimization Metrics
Report scheduled assets, average wait from planning start to maintenance start, time-weighted fleet availability, crew-pool utilization, planned completion rate, train-exposure proxy, and weighted objective score. Availability and completion metrics are planning proxies, not observed operational outcomes.

In [ ]:
scheduled_count = len(scheduled_rows)
request_count = len(maintenance_requests)
total_assets = len(optimization_df)
scheduled_hours = float(scheduled_rows["Duration Hours"].sum()) if scheduled_count else 0.0
average_delay_hours = float(scheduled_rows["Start Offset Hours"].mean()) if scheduled_count else 0.0
asset_availability_pct = (
    100.0 * (1.0 - scheduled_hours / (total_assets * HORIZON_HOURS))
    if total_assets and HORIZON_HOURS else 0.0
)
crew_utilization_pct = (
    100.0 * scheduled_hours / (MAX_CREWS_PER_SHIFT * HORIZON_HOURS)
    if MAX_CREWS_PER_SHIFT and HORIZON_HOURS else 0.0
)
completion_rate_pct = 100.0 * scheduled_count / request_count if request_count else 0.0
total_train_impacts = int(scheduled_rows["Estimated Train Impacts"].sum()) if scheduled_count else 0
optimization_score = float(solver.ObjectiveValue())

optimization_metrics = pd.DataFrame(
    [
        {"Metric": "Total Assets Scheduled", "Value": scheduled_count},
        {"Metric": "Average Delay (hours to start)", "Value": average_delay_hours},
        {"Metric": "Asset Availability (%)", "Value": asset_availability_pct},
        {"Metric": "Crew Utilization (%)", "Value": crew_utilization_pct},
        {"Metric": "Maintenance Completion Rate (%)", "Value": completion_rate_pct},
        {"Metric": "Estimated Train Impacts (proxy)", "Value": total_train_impacts},
        {"Metric": "Optimization Score", "Value": optimization_score},
        {"Metric": "Eligible Requests", "Value": request_count},
        {"Metric": "Solver Status", "Value": solver_status_name},
        {"Metric": "Solver Time (seconds)", "Value": solver_seconds},
    ]
)
display(optimization_metrics.style.format({"Value": "{:,.2f}"}))

---
## Section 11: Business Benefits
The report translates the schedule into operational terms. Train disruption is targeted through overnight preference, peak-hour penalties, train-frequency exposure, and an explicit penalty when no alternate track is available. Because this notebook has no historical or counterfactual schedule, it will report the estimated exposure and explain the mechanism without claiming a measured percentage reduction. Availability and completion are model-based planning estimates, not observed post-maintenance results.

In [ ]:
scheduled_asset_names = scheduled_rows["Asset"].astype(str).tolist() if scheduled_count else []
unscheduled_count = request_count - scheduled_count
business_report = f"""# RailOpt AI Maintenance Block Optimization Report

## Optimized plan
- Eligible assets considered: {total_assets:,}
- Maintenance requests above priority threshold: {request_count:,}
- Requests scheduled by CP-SAT: {scheduled_count:,}
- Requests not scheduled within the horizon/constraints: {unscheduled_count:,}
- Solver status: {solver_status_name}

## Operational interpretation
- Train disruption was targeted by preferring overnight blocks, penalizing passenger-peak overlap, weighting estimated train exposure by service frequency, and penalizing jobs without an alternate track. Estimated train exposure for this plan is {total_train_impacts:,} train-count proxy units. No historical baseline was available, so a measured reduction is not claimed.
- Planned crew utilization is {crew_utilization_pct:.1f}% across the configured pool of {MAX_CREWS_PER_SHIFT} crews. The objective rewards deployed crew-hours while respecting crew non-overlap.
- Estimated asset availability over the {HORIZON_DAYS}-day horizon is {asset_availability_pct:.2f}%, calculated as one minus scheduled asset-hours divided by all asset-hours. This is a planning proxy, not measured field availability.
- Planned maintenance completion rate is {completion_rate_pct:.1f}% of eligible requests. Unscheduled critical jobs should be escalated for human review.

## Operational caveats
- Station is used as a conservative proxy for a shared track conflict group because the source data has no track identifier.
- Crew count, work windows, passenger peaks, objective weights, and duration units must be confirmed by local operations before deployment.
- The plan is a decision-support schedule. Dispatch and safety rules remain authoritative.
"""
print(business_report)

---
## Section 12: Save Results
Save the full optimized request disposition, input request register, and KPI summary to `outputs/reports/`.

In [ ]:
schedule_path = REPORTS_DIR / "optimized_schedule.csv"
requests_path = REPORTS_DIR / "maintenance_requests.csv"
summary_path = REPORTS_DIR / "optimization_summary.csv"
business_report_path = REPORTS_DIR / "optimization_business_report.md"

optimized_schedule.to_csv(schedule_path, index=False)
maintenance_requests[request_columns + ["Critical"]].to_csv(requests_path, index=False)
optimization_metrics.to_csv(summary_path, index=False)
business_report_path.write_text(business_report, encoding="utf-8")
print(f"Saved schedule: {schedule_path}")
print(f"Saved requests: {requests_path}")
print(f"Saved summary: {summary_path}")
print(f"Saved business report: {business_report_path}")
print("Optimization stage complete. The schedule was generated by OR-Tools CP-SAT; the ML model was not retrained.")